# Preparing Student Score Records

*Clean two supplied score batches, retain quality evidence, and verify saved reports.*


## The Prepared Score Batches

The two original cleaning batches contain eight text records for four students.
Each record describes a student–subject score; later records replace earlier
versions of the same pair. Names are unique within this prepared case, and the
punctuation in `J!iyoung` is known entry noise. A real student system should use
stable IDs rather than cleaned names as keys.

`NA` and `None` mean missing. Scores must be whole numbers from 0 through 100.
Unavailable or invalid scores remain missing for analysis, with their reason
retained. No missing score is converted to zero, and invalid grades are not
clipped. A department lookup adds context. Reports describe only valid observed
grades and show their counts alongside their means.


## Imports, Raw Data, and Output Paths

All inputs are defined here. The output directory is relative to this chapter
locally and to `/home/pyodide` in the course site's browser filesystem.
Fixed filenames are replaced on each run.


In [1]:
import csv
import os
import pandas as pd

raw_batches = [
    [
        "  J!iyoung , 100 , mathematics",
        "Hojun, NA , English",
        "Hana , 85 , English ",
        "Kihoon, -3 , Mathematics",
    ],
    [
        "Jiyoung, 98 , Mathematics",
        " Hojun , 105 , Mathematics",
        "Hana, None , mathematics",
        "Kihoon , 86 , English ",
    ],
]
department_df = pd.DataFrame(
    {"subject": ["English", "Mathematics"], "department": ["Languages", "STEM"]}
)
output_directory = os.path.join("outputs", "08_student_records")
records_output_path = os.path.join(output_directory, "cleaned_scores.csv")
report_output_path = os.path.join(output_directory, "subject_report.csv")
os.makedirs(output_directory, exist_ok=True)
print("Raw batches:")
for batch_number, batch in enumerate(raw_batches, start=1):
    print("Batch", batch_number, batch)


Raw batches:
Batch 1 ['  J!iyoung , 100 , mathematics', 'Hojun, NA , English', 'Hana , 85 , English ', 'Kihoon, -3 , Mathematics']
Batch 2 ['Jiyoung, 98 , Mathematics', ' Hojun , 105 , Mathematics', 'Hana, None , mathematics', 'Kihoon , 86 , English ']


Batch 2 updates Jiyoung's Mathematics grade from 100 to 98. It also contains
the text `None` and an out-of-range 105, so batch order alone does not establish
data quality.


## Parse and Normalize While Preserving the Raw Fields

`prepare_records()` attaches a source order and keeps raw name, subject, and
score text. Nullable integers preserve missing scores. Status labels distinguish
missing input from a format or range failure. The accepted subject labels are
known from the prepared batches.


In [2]:
def prepare_records(batches):
    rows = []
    for batch in batches:
        for raw_name, raw_score, raw_subject in csv.reader(batch):
            rows.append(
                {
                    "source_order": len(rows) + 1,
                    "raw_name": raw_name,
                    "raw_subject": raw_subject,
                    "raw_score": raw_score,
                }
            )
    records = pd.DataFrame(rows)
    records["name"] = (
        records["raw_name"]
        .str.replace(r"[^A-Za-z ]", "", regex=True)
        .str.strip()
        .str.title()
    )
    records["subject"] = records["raw_subject"].str.strip().str.title()
    tokens = records["raw_score"].astype("string").str.strip()
    missing = tokens.isna() | tokens.str.lower().isin(["", "na", "none"])
    numeric = pd.to_numeric(tokens.mask(missing), errors="coerce").astype("Int64")
    format_error = ~missing & numeric.isna()
    range_error = numeric.notna() & ~numeric.between(0, 100)
    records["status"] = "valid"
    records.loc[missing, "status"] = "missing"
    records.loc[format_error, "status"] = "format_error"
    records.loc[range_error, "status"] = "range_error"
    records["score"] = numeric.mask(range_error)
    return records


prepared_df = prepare_records(raw_batches)
print(
    prepared_df[
        ["source_order", "name", "subject", "raw_score", "score", "status"]
    ].to_string(index=False)
)


 source_order    name     subject raw_score  score      status
            1 Jiyoung Mathematics      100     100       valid
            2   Hojun     English       NA    <NA>     missing
            3    Hana     English       85      85       valid
            4  Kihoon Mathematics       -3    <NA> range_error
            5 Jiyoung Mathematics       98      98       valid
            6   Hojun Mathematics      105    <NA> range_error
            7    Hana Mathematics     None    <NA>     missing
            8  Kihoon     English       86      86       valid


The eight prepared rows retain two missing inputs and two range errors. Four valid values appear before the repeated Jiyoung record is resolved.


## Retain the Latest Record and Attach Departments

Resolve versions by the normalized student–subject key before calculating
summaries. A left merge checks the lookup's key uniqueness. Sorting creates a
stable display and save order.


In [3]:
def retain_and_enrich(prepared, departments):
    latest = prepared.sort_values("source_order").drop_duplicates(
        ["name", "subject"], keep="last"
    )
    enriched = latest.merge(
        departments, on="subject", how="left", validate="many_to_one"
    )
    return enriched.sort_values(["name", "subject"]).reset_index(drop=True)


cleaned_df = retain_and_enrich(prepared_df, department_df)
print(
    cleaned_df[["name", "subject", "score", "status", "department"]].to_string(
        index=False
    )
)
print("Raw rows and retained keys:", len(prepared_df), len(cleaned_df))


   name     subject  score      status department
   Hana     English     85       valid  Languages
   Hana Mathematics   <NA>     missing       STEM
  Hojun     English   <NA>     missing  Languages
  Hojun Mathematics   <NA> range_error       STEM
Jiyoung Mathematics     98       valid       STEM
 Kihoon     English     86       valid  Languages
 Kihoon Mathematics   <NA> range_error       STEM
Raw rows and retained keys: 8 7


Seven keys remain. Jiyoung's retained score is 98. Three valid scores,
two missing inputs, and two range errors remain visible in the long table.


## Summaries Report Their Available Counts

`size` counts records, while `count` excludes missing scores. Means use only
valid observed scores. A wide view is convenient for reading but cannot by
itself distinguish an unrecorded subject from an invalid or missing grade;
the long table retains that evidence.


In [4]:
def build_reports(records):
    summary = records.groupby("subject", as_index=False).agg(
        record_count=("name", "size"),
        valid_score_count=("score", "count"),
        mean_score=("score", "mean"),
    )
    wide = records.pivot(index="name", columns="subject", values="score")
    return summary, wide


subject_report_df, wide_scores_df = build_reports(cleaned_df)
print("Subject report:")
print(subject_report_df.to_string(index=False))
print("Wide score view:")
print(wide_scores_df)


Subject report:
    subject  record_count  valid_score_count  mean_score
    English             3                  2        85.5
Mathematics             4                  1        98.0
Wide score view:
subject  English  Mathematics
name                         
Hana          85         <NA>
Hojun       <NA>         <NA>
Jiyoung     <NA>           98
Kihoon        86         <NA>


English has two valid scores across three records, with mean 85.5.
Mathematics has one valid score across four records, with mean 98. These counts
do not support a comparison of general subject difficulty.


## Validate, Save, and Reopen

The validation accepts explicitly unavailable grades while requiring every
available grade to satisfy the range rule. CSV stores values rather than pandas
types, so the reload specifies nullable scores and raw text. `keep_default_na=False`
preserves literal raw tokens such as `NA`; `na_values` applies only to the
cleaned score column. The report reload restores nullable types for its valid-score count and mean.


In [5]:
def save_and_verify(records, report):
    checks = {
        "unique_keys": not records.duplicated(["name", "subject"]).any(),
        "lookup_complete": bool(records["department"].notna().all()),
        "valid_score_range": bool(records["score"].dropna().between(0, 100).all()),
        "status_matches_availability": records["status"]
        .eq("valid")
        .equals(records["score"].notna()),
    }
    print("Validation:", checks)
    if not all(checks.values()):
        raise ValueError("Prepared records failed the declared validation rules")

    records.to_csv(records_output_path, index=False, encoding="utf-8")
    report.to_csv(report_output_path, index=False, encoding="utf-8")
    reopened_records = pd.read_csv(
        records_output_path,
        encoding="utf-8",
        keep_default_na=False,
        na_values={"score": [""]},
        dtype={"score": "Int64", "raw_score": "str"},
    )
    reopened_report = pd.read_csv(
        report_output_path,
        encoding="utf-8",
        dtype={"valid_score_count": "Int64", "mean_score": "Float64"},
    )
    print("Saved records match:", reopened_records.equals(records))
    print("Saved report matches:", reopened_report.equals(report))
    print("Saved files:", records_output_path, report_output_path)
    return reopened_records, reopened_report


## Run the Complete Case

`main()` begins from the raw batches, performs the same transformations, and
replaces the same two output files. Repeated runs cannot accumulate records.


In [6]:
def main():
    prepared = prepare_records(raw_batches)
    records = retain_and_enrich(prepared, department_df)
    report, wide = build_reports(records)
    print("Retained status counts:")
    print(records["status"].value_counts().sort_index())
    print("Subject report:")
    print(report.to_string(index=False))
    save_and_verify(records, report)
    return records, report, wide


final_records_df, final_report_df, final_wide_df = main()


Retained status counts:
status
missing        2
range_error    2
valid          3
Name: count, dtype: int64
Subject report:
    subject  record_count  valid_score_count  mean_score
    English             3                  2        85.5
Mathematics             4                  1        98.0
Validation: {'unique_keys': True, 'lookup_complete': True, 'valid_score_range': True, 'status_matches_availability': True}
Saved records match: True
Saved report matches: True
Saved files: outputs/08_student_records/cleaned_scores.csv outputs/08_student_records/subject_report.csv


## Result Analysis

Eight raw rows produce **seven unique student–subject records** for four students.
The retained records contain **three valid scores, two missing inputs, and two
range errors**. Jiyoung's later Mathematics score is **98**. The valid English
scores 85 and 86 have mean **85.5**; the only valid Mathematics score is 98.

All four validation rules pass. Both saved-table comparisons are `True`.
The saved long table retains raw text, source order, status, and department,
so an unavailable score is not confused with an observed zero. Repeating the
workflow produces the same records and replaces the same CSV files.
